In [ ]:
import torch
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from dataclasses import dataclass
# from utils import (
#   load_checkpoint,
#   save_checkpoint,
#   get_loaders,
#   check_accuracy,
#   save_predictions_as_imgs
# )

In [ ]:
@dataclass
class TrainerConfig:
  learning_rate: float = 1e-4
  device: str = "cuda" if torch.cuda.is_available() else "cpu"
  batch_size: int = 32
  num_epochs: int = 3
  num_workers: int = 2
  image_height: int = 160
  image_width: int = 240
  pin_memory: bool = True
  load_model: bool = False

In [ ]:
class Trainer():
  def __init__(self, trainer_loader, val_loader, model, optimizer, loss_fn, scaler, config=TrainerConfig()):
    self.trainer_loader = trainer_loader
    self.val_loader = val_loader
    self.model = model
    self.optimizer = optimizer
    self.loss_fn = loss_fn
    self.scaler = scaler
    self.config = config
    self.best_val_loss = float('inf')
    
  def __call__(self):
    for epoch in range(1, self.config.num_epochs + 1):
      train_loss = self.run_epoch(self.trainer_loader, training=True)
      val_loss = self.run_epoch(self.val_loader, training=False)
      print(f"Epoch [{epoch}/{self.config.num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

      if val_loss < self.best_val_loss:
        self.best_val_loss = val_loss
        torch.save(self.model.state_dict(), "best_model.pth")
        print("Best model saved!")

  def run_epoch(self, data_loader, training):
    self.model.train() if training else self.model.eval()
    loop = tqdm(data_loader, leave=True)
    running_loss = 0.0
    context = torch.enable_grad() if training else torch.no_grad()

    with context:
      for data, targets in loop:
        data = data.to(device=self.config.device)
        targets = targets.to(device=self.config.device)

        with torch.cuda.amp.autocast():
          predictions = self.model(data)
          loss = self.loss_fn(predictions, targets)

        if training:
          self.optimizer.zero_grad()
          self.scaler.scale(loss).backward()
          self.scaler.step(self.optimizer)
          self.scaler.update()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    return running_loss / len(data_loader)

      

2
